# unbranch

Split a HuggingFace repo with branched quantizations into separate single-BPW repos.

No large files are downloaded — only LFS pointers touch disk. HuggingFace's server resolves them.

---

<div style="background-color: #2d0000; border: 2px solid #ff4444; border-radius: 6px; padding: 12px 16px; margin: 12px 0;">
<strong style="color: #ff4444;">⚠ WARNING — DESTRUCTIVE OPERATIONS</strong>
<p style="color: #ff9999; margin: 8px 0 0 0;">This script performs <strong>irreversible</strong> operations: it renames your parent repo, force-pushes branches to main, and deletes branches. <strong>There is no undo.</strong> Always run with <code>DRY_RUN = True</code> first to preview what will happen.</p>
</div>

## Configuration

Fill in your details below, then run all cells.

In [ ]:
# ── Config ───────────────────────────────────────────────────
AUTHOR    = "UnstableLlama"
REPO_NAME = "Qwen3.5-4B-exl3"
BPWS      = [2.10, 3.00, 4.00, 5.00, 6.00]
DRY_RUN   = True   # Set to False when ready to execute
PRIVATE   = False  # Set to True to create new repos as private

## Authentication

In [ ]:
import getpass
import os

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HuggingFace Token: ")
assert HF_TOKEN, "A HuggingFace token is required"

## Setup

In [ ]:
import os
import sys
import time

from huggingface_hub import (
    CommitOperationAdd,
    CommitOperationCopy,
    CommitOperationDelete,
    HfApi,
    hf_hub_download,
)

api = HfApi(token=HF_TOKEN)

bpws = sorted(BPWS)
assert len(bpws) >= 2, "Need at least 2 BPW values (one becomes the renamed parent)"

largest_bpw = bpws[-1]
smaller_bpws = bpws[:-1]
parent_repo = f"{AUTHOR}/{REPO_NAME}"

HF_DELAY = 0.5


def fmt_bpw(bpw: float) -> str:
    return f"{bpw:.2f}"


def make_target_name(repo_name: str, bpw: float) -> str:
    return f"{repo_name}-{fmt_bpw(bpw)}bpw"


def target_id(bpw: float) -> str:
    return f"{AUTHOR}/{make_target_name(REPO_NAME, bpw)}"


def list_branch_files(repo_id, branch):
    items = list(api.list_repo_tree(
        repo_id=repo_id, revision=branch, recursive=True, repo_type="model",
    ))
    return [item for item in items if hasattr(item, "rfilename")]


def copy_branch_to_main(repo_id, branch, readme_text):
    """Server-side copy: overwrite main with branch content + updated README."""
    print(f"    Listing files on branch {branch}...")
    branch_files = list_branch_files(repo_id, branch)
    print(f"    Found {len(branch_files)} file(s)")

    operations = []
    for f in branch_files:
        if f.rfilename == "README.md":
            continue
        operations.append(CommitOperationCopy(
            src_path_in_repo=f.rfilename,
            path_in_repo=f.rfilename,
            src_revision=branch,
        ))

    operations.append(CommitOperationAdd(
        path_in_repo="README.md",
        path_or_fileobj=readme_text.encode(),
    ))

    main_files = list_branch_files(repo_id, "main")
    branch_filenames = {f.rfilename for f in branch_files} | {"README.md"}
    for f in main_files:
        if f.rfilename not in branch_filenames:
            operations.append(CommitOperationDelete(path_in_repo=f.rfilename))

    print(f"    Committing {len(operations)} operations to main...")
    api.create_commit(
        repo_id=repo_id, repo_type="model", operations=operations,
        commit_message=f"Copy {branch} to main and update README",
        revision="main",
    )


def save_main_as_backup(repo_id):
    backup = "_unbranch_backup_main"
    try:
        api.create_branch(repo_id, branch=backup, repo_type="model", revision="main")
    except Exception:
        try:
            api.delete_branch(repo_id, branch=backup, repo_type="model")
        except Exception:
            pass
        api.create_branch(repo_id, branch=backup, repo_type="model", revision="main")
    print(f"    Backup branch: {backup}")
    return backup


def restore_main_from_backup(repo_id, backup_branch):
    print(f"    Restoring main from {backup_branch}...")
    backup_files = list_branch_files(repo_id, backup_branch)
    current_files = list_branch_files(repo_id, "main")

    operations = []
    for f in backup_files:
        operations.append(CommitOperationCopy(
            src_path_in_repo=f.rfilename,
            path_in_repo=f.rfilename,
            src_revision=backup_branch,
        ))

    backup_filenames = {f.rfilename for f in backup_files}
    for f in current_files:
        if f.rfilename not in backup_filenames:
            operations.append(CommitOperationDelete(path_in_repo=f.rfilename))

    if operations:
        api.create_commit(
            repo_id=repo_id, repo_type="model", operations=operations,
            commit_message="Restore main from backup",
            revision="main",
        )
    print(f"    Main restored.")


def add_shared_files_to_main(repo_id, source_branch):
    """Copy files from source_branch into main that don't already exist."""
    source_files = list_branch_files(repo_id, source_branch)
    main_files = list_branch_files(repo_id, "main")
    main_filenames = {f.rfilename for f in main_files}

    operations = []
    for f in source_files:
        if f.rfilename not in main_filenames:
            operations.append(CommitOperationCopy(
                src_path_in_repo=f.rfilename,
                path_in_repo=f.rfilename,
                src_revision=source_branch,
            ))

    if operations:
        print(f"    Adding {len(operations)} shared file(s) from original main...")
        api.create_commit(
            repo_id=repo_id, repo_type="model", operations=operations,
            commit_message="Add shared files from original main",
            revision="main",
        )
    else:
        print(f"    No additional shared files to add.")


print("Target repos:")
for bpw in bpws:
    tag = "(renamed parent)" if bpw == largest_bpw else "(new)"
    print(f"  {target_id(bpw)}  {tag}")

## Step 1 — Download & rewrite README

The only file downloaded locally. Rewrites branch URLs to single-repo URLs.

In [ ]:
def rewrite_readme(readme, author, repo_name, bpws):
    parent = f"{author}/{repo_name}"
    for bpw in bpws:
        b = fmt_bpw(bpw)
        branch = f"{b}bpw"
        target = f"{author}/{make_target_name(repo_name, bpw)}"
        readme = readme.replace(f"{parent}/tree/{branch}", target)
        readme = readme.replace(f'{parent} --revision "{branch}"', target)
        readme = readme.replace(f"{parent} --revision {branch}", target)
    return readme


readme_file = hf_hub_download(parent_repo, "README.md", token=HF_TOKEN)
with open(readme_file) as f:
    original_readme = f.read()

readme_text = rewrite_readme(original_readme, AUTHOR, REPO_NAME, bpws)

changed = original_readme != readme_text
print(f"README modified: {changed}")
if changed:
    old_lines = set(original_readme.splitlines())
    new_lines = set(readme_text.splitlines())
    for line in sorted(new_lines - old_lines):
        if "huggingface.co" in line or "hf download" in line:
            print(f"  + {line.strip()[:120]}")

## Step 2 — Create repos for smaller BPWs

For each BPW except the largest:
1. Copy branch files → parent's main (server-side via `CommitOperationCopy`)
2. Add shared files from original main → parent's main
3. `duplicate_repo` → snapshot parent's main into new single-BPW repo
4. Restore parent's main from backup

No model files are downloaded. Everything happens via the HF API.

In [ ]:
largest_branch = f"{fmt_bpw(largest_bpw)}bpw"
largest_repo = target_id(largest_bpw)

if DRY_RUN:
    print("[DRY RUN] Would create:")
    for bpw in smaller_bpws:
        print(f"  {target_id(bpw)}  ← branch {fmt_bpw(bpw)}bpw")
else:
    # Back up parent's main so we can restore after each cycle
    print("Backing up parent main branch...")
    backup_branch = save_main_as_backup(parent_repo)
    time.sleep(HF_DELAY)

    for bpw in smaller_bpws:
        branch = f"{fmt_bpw(bpw)}bpw"
        repo_id = target_id(bpw)
        print(f"\n── {repo_id} (from branch {branch}) ──")

        # a. Copy branch → parent's main
        print(f"  Copying {branch} → main on parent...")
        copy_branch_to_main(parent_repo, branch, readme_text)
        time.sleep(HF_DELAY)

        # b. Add shared files from original main (so the duplicate gets everything)
        add_shared_files_to_main(parent_repo, backup_branch)
        time.sleep(HF_DELAY)

        # c. Check target doesn't already exist
        try:
            api.repo_info(repo_id, repo_type="model")
            print(f"\n  ERROR: {repo_id} already exists!")
            print(f"  If it's leftover from a failed run, delete it manually:")
            print(f"    https://huggingface.co/{repo_id}/settings")
            print(f"  Then re-run this script.")
            raise RuntimeError(f"{repo_id} already exists — aborting")
        except RuntimeError:
            raise
        except Exception:
            pass  # Repo doesn't exist — good

        # d. Duplicate parent (main now has BPW files + shared files) → new repo
        print(f"  Duplicating parent → {repo_id}")
        api.duplicate_repo(
            from_id=parent_repo, to_id=repo_id,
            private=PRIVATE, repo_type="model",
        )
        time.sleep(HF_DELAY)

        # e. Restore parent's main
        restore_main_from_backup(parent_repo, backup_branch)
        time.sleep(HF_DELAY)

        # f. Delete inherited branches from the new repo
        for other_bpw in bpws:
            try:
                api.delete_branch(repo_id, branch=f"{fmt_bpw(other_bpw)}bpw", repo_type="model")
                time.sleep(HF_DELAY)
            except Exception:
                pass
        try:
            api.delete_branch(repo_id, branch=backup_branch, repo_type="model")
        except Exception:
            pass

        print(f"  done.")

    # Rename backup branch to main_original on parent
    try:
        api.create_branch(
            parent_repo, branch="main_original",
            repo_type="model", revision=backup_branch,
        )
        api.delete_branch(parent_repo, branch=backup_branch, repo_type="model")
        print(f"\nPreserved original main as branch 'main_original'")
    except Exception as e:
        print(f"\nCould not rename backup branch: {e}")
        print(f"Original main is still on branch '{backup_branch}'")

## Step 3 — Copy largest BPW to parent's main & rename

Copies the largest branch to main via API, then renames the parent repo. Preserves download counts.

In [ ]:
if DRY_RUN:
    print(f"[DRY RUN] Would copy {largest_branch} → main on {parent_repo}")
    print(f"[DRY RUN] Would rename {parent_repo} → {largest_repo}")
else:
    print(f"Copying {largest_branch} → main on {parent_repo} (via API)")
    copy_branch_to_main(parent_repo, largest_branch, readme_text)
    time.sleep(HF_DELAY)

    # Add shared files from original main
    add_shared_files_to_main(parent_repo, "main_original")
    time.sleep(HF_DELAY)

    print(f"\nRenaming {parent_repo} → {largest_repo}")
    api.move_repo(from_id=parent_repo, to_id=largest_repo, repo_type="model")
    print("  done.")
    time.sleep(HF_DELAY)

## Step 4 — Verify

Check that all repos exist and have model files.

In [ ]:
all_ok = True

if DRY_RUN:
    for bpw in bpws:
        print(f"  [DRY RUN] Would verify {target_id(bpw)}")
else:
    for bpw in bpws:
        repo_id = target_id(bpw)
        try:
            files = api.list_repo_files(repo_id, repo_type="model")
            count = len(files)
            has_safetensors = any(f.endswith(".safetensors") for f in files)
            status = "OK" if count >= 2 and has_safetensors else "CHECK"
            print(f"  {repo_id}: {count} files [{status}]")
            if count < 2 or not has_safetensors:
                all_ok = False
        except Exception as e:
            print(f"  {repo_id}: ERROR — {e}")
            all_ok = False
        time.sleep(HF_DELAY)

    if not all_ok:
        print("\n  Some repos may have issues. Fix before running Step 5.")
        raise RuntimeError("Verification failed — do NOT proceed to Step 5")

## Step 5 — Delete old branches

Only run this after verifying all repos in Step 4. Removes the old BPW branches from the renamed parent repo.

In [ ]:
if DRY_RUN:
    print("[DRY RUN] Would delete these branches from", largest_repo)
    for bpw in bpws:
        print(f"  {fmt_bpw(bpw)}bpw")
else:
    for bpw in bpws:
        branch = f"{fmt_bpw(bpw)}bpw"
        try:
            api.delete_branch(largest_repo, branch=branch, repo_type="model")
            print(f"  Deleted: {branch}")
            time.sleep(HF_DELAY)
        except Exception as e:
            print(f"  Could not delete {branch}: {e}")

print(f"\n{'=' * 60}")
if DRY_RUN:
    print("Dry run complete. Set DRY_RUN = False to execute for real.")
else:
    print(f"Done! Created {len(bpws)} single-BPW repos:")
for bpw in bpws:
    print(f"  https://huggingface.co/{target_id(bpw)}")
print(f"{'=' * 60}")